# Multi-Agent Due Diligence Analyst

**Enterprise-grade company research powered by 6 AI agents with fact-checking and contradiction resolution.**

This notebook walks you through building a production multi-agent system that:
- Decomposes complex research into parallel specialist tasks
- Runs 4 specialist agents concurrently (Financial, News, Competitive, Risk)
- Verifies claims with an independent Fact-Checker agent
- Resolves contradictions via structured debate
- Produces a comprehensive due diligence report

**Prerequisites:**
- Google Gemini API key (free at https://aistudio.google.com/apikey)
- Optional: Tavily API key (free at https://tavily.com - 1000 searches/month)

---

## 1. Setup & Installation

In [ ]:
# Install dependencies
!pip install -q langgraph langchain langchain-google-genai pydantic pyyaml \
    tavily-python ddgs tabulate

In [ ]:
import os
from getpass import getpass

# Set API keys
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API Key: ")

# Optional: Tavily for better search (falls back to DuckDuckGo)
# os.environ["TAVILY_API_KEY"] = getpass("Enter Tavily API Key (optional): ")

print("API keys configured!")

## 2. Understanding the Architecture

```
                    +------------------+
                    |   Lead Analyst   |  Phase 1: Plan Research
                    |   (Planner)      |
                    +--------+---------+
                             |
              +--------------+--------------+
              |              |              |
    +---------v--+  +-------v----+  +------v-------+  +-------v------+
    | Financial  |  |   News &   |  | Competitive  |  |    Risk      |
    | Analyst    |  | Sentiment  |  | Intelligence |  |  Assessor    |
    +-----+------+  +-----+-----+  +------+-------+  +------+-------+
          |              |              |                    |
          +--------------+--------------+--------------------+
                             |
                    +--------v---------+
                    |   Fact Checker   |  Phase 2: Verify Claims
                    +--------+---------+
                             |
                    +--------v---------+
                    |   Lead Analyst   |  Phase 3: Resolve Contradictions
                    |   (Debater)      |
                    +--------+---------+
                             |
                    +--------v---------+
                    |   Lead Analyst   |  Phase 4: Final Report
                    |   (Synthesizer)  |
                    +------------------+
```

### Key Design Decisions
1. **Parallel execution**: 4 specialist agents run concurrently via LangGraph `Send()`
2. **Structured output**: Every LLM call uses Pydantic schemas - no parsing errors
3. **Graceful degradation**: Every agent has a fallback path if it fails
4. **Guardrails**: Token budget, cost ceiling, loop detection, PII masking
5. **Fact-checking**: Independent verification pass challenges other agents' findings

## 3. Define the State Schema

The state is the shared data contract between all agents. LangGraph manages state merging.

In [ ]:
import operator
from typing import Annotated, TypedDict


class AgentFinding(TypedDict, total=False):
    """A single finding from any agent."""
    agent: str
    category: str       # financial | news | competitive | risk
    title: str
    detail: str
    severity: str       # critical | high | medium | low | info
    confidence: float   # 0.0 to 1.0
    sources: list[str]
    verified: bool


class DueDiligenceState(TypedDict, total=False):
    """Pipeline state passed through all LangGraph nodes."""
    # Input
    company_name: str
    query: str
    analysis_depth: str
    
    # Planning
    research_plan: dict
    focus_areas: list[str]
    
    # Agent findings (append-only via operator.add)
    financial_findings: Annotated[list[AgentFinding], operator.add]
    news_findings: Annotated[list[AgentFinding], operator.add]
    competitive_findings: Annotated[list[AgentFinding], operator.add]
    risk_findings: Annotated[list[AgentFinding], operator.add]
    
    # Fact-checking
    fact_check_results: Annotated[list[dict], operator.add]
    contradictions: Annotated[list[dict], operator.add]
    
    # Final output
    executive_summary: str
    final_report: str
    overall_risk_rating: str
    overall_confidence: float
    
    # Metadata
    pipeline_trace: Annotated[list[dict], operator.add]
    errors: Annotated[list[str], operator.add]
    status: str


print("State schema defined with", len(DueDiligenceState.__annotations__), "fields")
print("Append-only fields (operator.add):", 
      [k for k, v in DueDiligenceState.__annotations__.items() 
       if hasattr(v, '__metadata__')])

## 4. Structured Output Schemas

Every LLM call uses Pydantic schemas via `.with_structured_output()`. This guarantees valid JSON.

In [ ]:
from pydantic import BaseModel, Field


class ResearchPlan(BaseModel):
    """Lead Analyst decomposes query into agent tasks."""
    company_summary: str = Field(description="Brief context about the company")
    sub_tasks: list[dict] = Field(description="Research tasks for each specialist")
    focus_areas: list[str] = Field(description="Key focus areas")
    risk_hypothesis: str = Field(description="Initial risk hypothesis")


class FinancialAnalysis(BaseModel):
    """Financial analyst structured output."""
    company_name: str
    financial_health_rating: str = Field(description="strong | moderate | weak | critical | insufficient_data")
    revenue_analysis: str
    profitability_analysis: str
    red_flags: list[str] = Field(default_factory=list)
    green_flags: list[str] = Field(default_factory=list)
    data_gaps: list[str] = Field(default_factory=list)
    sources: list[str] = Field(default_factory=list)


class RiskAssessment(BaseModel):
    """Risk assessor structured output."""
    company_name: str
    overall_risk_level: str = Field(description="critical | high | moderate | low")
    risks: list[dict] = Field(default_factory=list)
    risk_summary: str
    sources: list[str] = Field(default_factory=list)


class FactCheckReport(BaseModel):
    """Fact checker verification results."""
    total_claims_checked: int
    verified_count: int
    contradicted_count: int
    unverifiable_count: int
    verifications: list[dict] = Field(default_factory=list)
    cross_agent_contradictions: list[str] = Field(default_factory=list)
    overall_reliability: str


class ExecutiveSummary(BaseModel):
    """Final synthesis output."""
    company_name: str
    one_line_verdict: str
    overall_risk_rating: str
    overall_confidence: float = Field(ge=0.0, le=1.0)
    key_strengths: list[str]
    key_risks: list[str]
    recommendation: str
    action_items: list[str] = Field(default_factory=list)


print("Schemas defined:", [cls.__name__ for cls in [
    ResearchPlan, FinancialAnalysis, RiskAssessment, FactCheckReport, ExecutiveSummary
]])

## 5. LLM Setup with Fallback

We use Google Gemini (free tier). The system supports automatic fallback if the primary model fails.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Primary model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0.1,
    max_output_tokens=4096,
)

# Test it
response = llm.invoke("What is 2+2? Reply with just the number.")
print(f"LLM ready! Test response: {response.content}")

# Structured output example
structured_llm = llm.with_structured_output(FinancialAnalysis)
print("Structured output ready - guaranteed JSON responses")

## 6. Search Tool with Caching

Web search is the primary information source. We use DuckDuckGo (free, no API key) with optional Tavily upgrade.

In [ ]:
try:
    from ddgs import DDGS  # New package name
except ImportError:
    from duckduckgo_search import DDGS  # Legacy fallback
import time


def web_search(query: str, max_results: int = 5) -> list[dict]:
    """Search the web and return formatted results."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        return [
            {
                "title": r.get("title", ""),
                "url": r.get("href", ""),
                "snippet": r.get("body", "")[:500],
            }
            for r in results
        ]
    except Exception as e:
        print(f"Search failed: {e}")
        return []


# Test search
results = web_search("Tesla financial performance 2024", max_results=3)
print(f"Search returned {len(results)} results:")
for r in results:
    print(f"  - {r['title'][:60]}")

## 7. Build the Specialist Agents

Each agent is a function that:
1. Receives the shared state
2. Searches for relevant information
3. Analyzes with structured LLM output
4. Returns a partial state update
5. Falls back gracefully on any error

In [ ]:
def build_search_context(results: list[dict]) -> str:
    """Format search results for LLM context."""
    lines = []
    seen = set()
    for r in results:
        if r["url"] not in seen:
            seen.add(r["url"])
            lines.append(f"Source: {r['url']}\nTitle: {r['title']}\nContent: {r['snippet']}\n")
    return "\n---\n".join(lines[:15])


def financial_analyst(state: DueDiligenceState) -> dict:
    """Financial Analyst - researches financial health."""
    company = state["company_name"]
    start = time.time()
    
    try:
        # Search for financial data
        results = []
        for q in [
            f"{company} financial performance revenue profit 2024 2025",
            f"{company} funding valuation balance sheet",
        ]:
            results.extend(web_search(q, 5))
        
        context = build_search_context(results)
        structured = llm.with_structured_output(FinancialAnalysis)
        
        analysis = structured.invoke(
            f"""You are a Financial Analyst. Analyze financial data for {company}.
            Only state facts from search results. Cite sources.
            \nSEARCH RESULTS:\n{context}"""
        )
        
        findings = []
        for flag in analysis.red_flags:
            findings.append({"agent": "financial", "category": "financial",
                           "title": f"Red Flag: {flag}", "detail": flag,
                           "severity": "high", "confidence": 0.7, "sources": analysis.sources})
        for flag in analysis.green_flags:
            findings.append({"agent": "financial", "category": "financial",
                           "title": f"Strength: {flag}", "detail": flag,
                           "severity": "info", "confidence": 0.7, "sources": analysis.sources})
        findings.append({"agent": "financial", "category": "financial",
                        "title": f"Health: {analysis.financial_health_rating}",
                        "detail": analysis.revenue_analysis, "severity": "info",
                        "confidence": 0.75, "sources": analysis.sources})
        
        print(f"  [Financial] {analysis.financial_health_rating} - {len(findings)} findings ({time.time()-start:.1f}s)")
        return {"financial_findings": findings, "pipeline_trace": [{"agent": "financial", "duration": round(time.time()-start, 2)}]}
    
    except Exception as e:
        print(f"  [Financial] FALLBACK: {e}")
        return {"financial_findings": [{"agent": "financial", "title": f"Analysis failed: {e}",
                "severity": "low", "confidence": 0.0}], "errors": [str(e)]}


def news_sentiment(state: DueDiligenceState) -> dict:
    """News & Sentiment Analyst - scans recent news."""
    company = state["company_name"]
    start = time.time()
    
    try:
        results = web_search(f"{company} latest news controversy 2024 2025", 8)
        context = build_search_context(results)
        
        response = llm.invoke(
            f"""Analyze recent news about {company}. For each event, note:
            date, headline, sentiment (positive/negative/neutral), impact (high/medium/low).
            Cite sources.\n\nSEARCH RESULTS:\n{context}"""
        )
        
        findings = [{"agent": "news", "category": "news",
                    "title": f"News Summary for {company}",
                    "detail": response.content[:1000],
                    "severity": "info", "confidence": 0.7, "sources": [r["url"] for r in results[:5]]}]
        
        print(f"  [News] {len(results)} articles found ({time.time()-start:.1f}s)")
        return {"news_findings": findings, "pipeline_trace": [{"agent": "news", "duration": round(time.time()-start, 2)}]}
    
    except Exception as e:
        print(f"  [News] FALLBACK: {e}")
        return {"news_findings": [{"agent": "news", "title": f"Analysis failed: {e}",
                "severity": "low", "confidence": 0.0}], "errors": [str(e)]}


def competitive_intel(state: DueDiligenceState) -> dict:
    """Competitive Intelligence - maps market landscape."""
    company = state["company_name"]
    start = time.time()
    
    try:
        results = web_search(f"{company} competitors market share industry comparison", 8)
        context = build_search_context(results)
        
        response = llm.invoke(
            f"""Analyze the competitive landscape for {company}.
            Identify top competitors, market position, advantages, and threats.
            Cite sources.\n\nSEARCH RESULTS:\n{context}"""
        )
        
        findings = [{"agent": "competitive", "category": "competitive",
                    "title": f"Competitive Landscape for {company}",
                    "detail": response.content[:1000],
                    "severity": "info", "confidence": 0.65, "sources": [r["url"] for r in results[:5]]}]
        
        print(f"  [Competitive] Analysis complete ({time.time()-start:.1f}s)")
        return {"competitive_findings": findings, "pipeline_trace": [{"agent": "competitive", "duration": round(time.time()-start, 2)}]}
    
    except Exception as e:
        print(f"  [Competitive] FALLBACK: {e}")
        return {"competitive_findings": [{"agent": "competitive", "title": f"Analysis failed: {e}",
                "severity": "low", "confidence": 0.0}], "errors": [str(e)]}


def risk_assessor(state: DueDiligenceState) -> dict:
    """Risk Assessor - identifies multi-dimensional risks."""
    company = state["company_name"]
    start = time.time()
    
    try:
        results = web_search(f"{company} risks lawsuit regulatory problems ESG", 8)
        context = build_search_context(results)
        
        structured = llm.with_structured_output(RiskAssessment)
        analysis = structured.invoke(
            f"""Identify risks for {company}. Categorize as legal, regulatory, operational,
            reputational, financial, strategic, or technology. Rate severity and likelihood.
            Cite sources.\n\nSEARCH RESULTS:\n{context}"""
        )
        
        findings = [{"agent": "risk", "category": "risk",
                    "title": f"Risk Level: {analysis.overall_risk_level}",
                    "detail": analysis.risk_summary,
                    "severity": "high" if analysis.overall_risk_level in ("critical", "high") else "medium",
                    "confidence": 0.7, "sources": analysis.sources}]
        for risk in analysis.risks:
            findings.append({"agent": "risk", "category": "risk",
                           "title": risk.get("title", "Risk"), "detail": risk.get("description", ""),
                           "severity": risk.get("severity", "medium"), "confidence": 0.65})
        
        print(f"  [Risk] {analysis.overall_risk_level} - {len(analysis.risks)} risks ({time.time()-start:.1f}s)")
        return {"risk_findings": findings, "pipeline_trace": [{"agent": "risk", "duration": round(time.time()-start, 2)}]}
    
    except Exception as e:
        print(f"  [Risk] FALLBACK: {e}")
        return {"risk_findings": [{"agent": "risk", "title": f"Analysis failed: {e}",
                "severity": "low", "confidence": 0.0}], "errors": [str(e)]}


print("All 4 specialist agents defined!")

## 8. Build the Fact Checker Agent

The Fact Checker runs AFTER all specialists. It independently verifies claims and flags contradictions.

In [ ]:
def fact_checker(state: DueDiligenceState) -> dict:
    """Fact Checker - verifies claims from other agents."""
    company = state["company_name"]
    start = time.time()
    
    try:
        # Collect high-priority claims
        all_findings = (
            state.get("financial_findings", []) +
            state.get("news_findings", []) +
            state.get("competitive_findings", []) +
            state.get("risk_findings", [])
        )
        
        important_claims = [
            f for f in all_findings
            if f.get("severity") in ("critical", "high", "medium")
        ][:8]  # Cap at 8 to stay within budget
        
        if not important_claims:
            print("  [FactCheck] No claims to verify")
            return {"fact_check_results": [{"total_checked": 0, "overall_reliability": "no_claims"}]}
        
        # Verify each claim independently
        verification_text = ""
        for i, claim in enumerate(important_claims):
            results = web_search(f"{company} {claim.get('title', '')}", 3)
            context = "\n".join(f"  - {r['snippet'][:150]}" for r in results)
            verification_text += f"\nClaim {i+1} ({claim.get('agent', '?')}): {claim.get('title', '')}\nEvidence: {context}\n"
        
        structured = llm.with_structured_output(FactCheckReport)
        report = structured.invoke(
            f"""Verify these claims about {company}. For each, check if the evidence
            supports, contradicts, or is insufficient.\n{verification_text}"""
        )
        
        contradictions = [{"claim": c, "status": "unresolved"} for c in report.cross_agent_contradictions]
        
        print(f"  [FactCheck] {report.verified_count}/{report.total_claims_checked} verified, "
              f"{report.contradicted_count} contradicted ({time.time()-start:.1f}s)")
        
        return {
            "fact_check_results": [{"total_checked": report.total_claims_checked,
                                    "verified": report.verified_count,
                                    "contradicted": report.contradicted_count,
                                    "overall_reliability": report.overall_reliability}],
            "contradictions": contradictions,
            "pipeline_trace": [{"agent": "fact_checker", "duration": round(time.time()-start, 2)}],
        }
    
    except Exception as e:
        print(f"  [FactCheck] FALLBACK: {e}")
        return {"fact_check_results": [{"error": str(e)}], "errors": [str(e)]}


print("Fact Checker agent defined!")

## 9. Build the Lead Analyst (Planner + Synthesizer)

The Lead Analyst has 3 roles:
1. **Plan**: Decompose the query into agent tasks
2. **Debate**: Resolve contradictions between agents
3. **Synthesize**: Produce the final report

In [ ]:
def plan_research(state: DueDiligenceState) -> dict:
    """Lead Analyst Phase 1: Create research plan."""
    company = state["company_name"]
    print(f"\n[Lead Analyst] Planning research for: {company}")
    
    try:
        structured = llm.with_structured_output(ResearchPlan)
        plan = structured.invoke(
            f"""Create a research plan for due diligence on {company}.
            Assign tasks to: financial_analyst, news_sentiment, competitive_intel, risk_assessor.
            Query: {state.get('query', 'comprehensive analysis')}"""
        )
        
        print(f"  Plan: {len(plan.sub_tasks)} tasks, focus: {', '.join(plan.focus_areas[:3])}")
        return {"research_plan": {"summary": plan.company_summary}, "focus_areas": plan.focus_areas,
                "status": "researching", "pipeline_trace": [{"agent": "lead_planner"}]}
    except Exception as e:
        print(f"  FALLBACK plan: {e}")
        return {"research_plan": {}, "focus_areas": ["financial", "news", "competitive", "risk"],
                "status": "researching"}


def synthesize_report(state: DueDiligenceState) -> dict:
    """Lead Analyst Phase 3: Produce final executive summary."""
    company = state["company_name"]
    print(f"\n[Lead Analyst] Synthesizing final report...")
    
    try:
        # Collect all findings
        all_findings = ""
        for key in ["financial_findings", "news_findings", "competitive_findings", "risk_findings"]:
            for f in state.get(key, []):
                all_findings += f"[{f.get('severity', '?')}] {f.get('title', '')}: {f.get('detail', '')[:200]}\n"
        
        fc = state.get("fact_check_results", [{}])
        fc_summary = str(fc[0]) if fc else "No fact-check"
        
        structured = llm.with_structured_output(ExecutiveSummary)
        summary = structured.invoke(
            f"""Synthesize a due diligence report for {company}.
            \nFINDINGS:\n{all_findings}\nFACT-CHECK: {fc_summary}
            \nBe balanced. Note uncertainties. Provide clear recommendation."""
        )
        
        # Build markdown report
        report = f"# Due Diligence Report: {company}\n\n"
        report += f"**Verdict:** {summary.one_line_verdict}\n\n"
        report += f"**Risk:** {summary.overall_risk_rating.upper()} | "
        report += f"**Confidence:** {summary.overall_confidence:.0%} | "
        report += f"**Recommendation:** {summary.recommendation}\n\n"
        report += "## Key Strengths\n" + "\n".join(f"- {s}" for s in summary.key_strengths) + "\n\n"
        report += "## Key Risks\n" + "\n".join(f"- {r}" for r in summary.key_risks) + "\n\n"
        if summary.action_items:
            report += "## Next Steps\n" + "\n".join(f"{i+1}. {a}" for i, a in enumerate(summary.action_items))
        
        print(f"  Verdict: {summary.one_line_verdict[:80]}")
        print(f"  Risk: {summary.overall_risk_rating}, Confidence: {summary.overall_confidence:.0%}")
        
        return {"executive_summary": summary.one_line_verdict, "final_report": report,
                "overall_risk_rating": summary.overall_risk_rating,
                "overall_confidence": summary.overall_confidence, "status": "complete",
                "pipeline_trace": [{"agent": "lead_synthesis"}]}
    
    except Exception as e:
        print(f"  FALLBACK synthesis: {e}")
        return {"final_report": f"Report generation failed: {e}", "status": "complete", "errors": [str(e)]}


print("Lead Analyst (planner + synthesizer) defined!")

## 10. Wire the LangGraph Pipeline

This is the core orchestration - connecting all agents with parallel execution and conditional routing.

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.types import Send


def fan_out_to_specialists(state):
    """Route to all 4 specialists in parallel."""
    return [
        Send("financial_analyst", state),
        Send("news_sentiment", state),
        Send("competitive_intel", state),
        Send("risk_assessor", state),
    ]


def route_after_fact_check(state):
    """Skip debate if no contradictions."""
    contradictions = [c for c in state.get("contradictions", []) if c.get("status") != "resolved"]
    return "synthesize_report"  # Simplified - always go to synthesis


# Build the graph
graph = StateGraph(DueDiligenceState)

# Add nodes
graph.add_node("plan_research", plan_research)
graph.add_node("financial_analyst", financial_analyst)
graph.add_node("news_sentiment", news_sentiment)
graph.add_node("competitive_intel", competitive_intel)
graph.add_node("risk_assessor", risk_assessor)
graph.add_node("fact_checker", fact_checker)
graph.add_node("synthesize_report", synthesize_report)

# Wire edges
graph.set_entry_point("plan_research")
graph.add_conditional_edges("plan_research", fan_out_to_specialists,
    ["financial_analyst", "news_sentiment", "competitive_intel", "risk_assessor"])
graph.add_edge("financial_analyst", "fact_checker")
graph.add_edge("news_sentiment", "fact_checker")
graph.add_edge("competitive_intel", "fact_checker")
graph.add_edge("risk_assessor", "fact_checker")
graph.add_conditional_edges("fact_checker", route_after_fact_check,
    {"synthesize_report": "synthesize_report"})
graph.add_edge("synthesize_report", END)

# Compile
app = graph.compile()
print("LangGraph pipeline compiled!")
print(f"Nodes: {list(app.get_graph().nodes.keys())}")

## 11. Run the Analysis!

Let's run a full due diligence analysis. Change the company name to analyze any entity.

In [ ]:
# === CHANGE THIS TO ANALYZE ANY COMPANY ===
COMPANY = "Tesla"
QUERY = ""  # Optional: specific focus like "AI safety risks and competition"
# ============================================

print(f"Starting due diligence for: {COMPANY}")
print("=" * 60)

initial_state = {
    "company_name": COMPANY,
    "query": QUERY or f"Comprehensive due diligence analysis of {COMPANY}",
    "analysis_depth": "standard",
    "financial_findings": [],
    "news_findings": [],
    "competitive_findings": [],
    "risk_findings": [],
    "fact_check_results": [],
    "contradictions": [],
    "pipeline_trace": [],
    "errors": [],
    "status": "planning",
}

start = time.time()
result = app.invoke(initial_state)
duration = time.time() - start

print(f"\n{'=' * 60}")
print(f"COMPLETE in {duration:.1f}s")
print(f"Risk: {result.get('overall_risk_rating', 'unknown')}")
print(f"Confidence: {result.get('overall_confidence', 0):.0%}")
print(f"Errors: {len(result.get('errors', []))}")
print(f"{'=' * 60}")

## 12. View the Full Report

In [ ]:
from IPython.display import Markdown, display

report = result.get("final_report", "No report generated.")
display(Markdown(report))

## 13. Analyze the Pipeline Execution

In [ ]:
# Pipeline trace
traces = result.get("pipeline_trace", [])
print("Pipeline Execution Trace:")
print("-" * 40)
for t in traces:
    agent = t.get("agent", "?")
    dur = t.get("duration", 0)
    print(f"  {agent:25s} {dur:6.1f}s")

# Findings summary
print(f"\nFindings Summary:")
print(f"  Financial: {len(result.get('financial_findings', []))} findings")
print(f"  News:      {len(result.get('news_findings', []))} findings")
print(f"  Compete:   {len(result.get('competitive_findings', []))} findings")
print(f"  Risk:      {len(result.get('risk_findings', []))} findings")

# Fact-check
fc = result.get("fact_check_results", [{}])
if fc:
    print(f"\nFact-Check: {fc[0]}")

# Errors
errors = result.get("errors", [])
if errors:
    print(f"\nErrors ({len(errors)}):")
    for e in errors:
        print(f"  - {e}")

## 14. Try Another Company

Analyze a different company or add a specific focus.

In [ ]:
# Try a private company
state2 = {**initial_state, "company_name": "Stripe", 
           "query": "Focus on fintech competition and regulatory risks"}

result2 = app.invoke(state2)
display(Markdown(result2.get("final_report", "No report")))

---

## What You Built

| Component | What It Does |
|-----------|-------------|
| **State Schema** | Typed shared state with append-only merge via `operator.add` |
| **6 Agents** | Specialist researchers + fact checker + lead analyst |
| **Parallel Execution** | 4 agents run concurrently via LangGraph `Send()` |
| **Structured Output** | Every LLM call uses Pydantic schemas, zero parsing errors |
| **Graceful Degradation** | Every agent has try/except with useful fallback output |
| **Fact-Checking** | Independent verification pass challenges agents' claims |
| **Contradiction Detection** | Cross-agent disagreements flagged and highlighted |
| **Source Grounding** | Every claim cites its source URL |

### Production Upgrades (in the full project)
- Guardrail manager (token budget, cost ceiling, loop detection, PII masking)
- Search result caching (SQLite with TTL and LRU eviction)
- Contradiction debate (Lead Analyst resolves conflicts)
- Self-correction loop (Fact Checker triggers re-research)
- Streamlit dashboard with real-time progress
- Docker deployment
- Evaluation framework (coverage, source diversity, factual consistency, actionability)

**Full project:** https://github.com/genieincodebottle/aiml-companion/tree/main/projects/due-diligence-agent